# Autotuning GAN

The aim of this project is to use a genetic algorithm approach to train a GAN (Generative Adversarial Network) in order to enhance the quality of a Deep Neural Network (DNN).

## Strategy

The primary approach is based on the Pix2Pix architecture, a type of GAN that incorporates residual connections between the encoder and decoder layers to enhance the quality of the generated images. This type is called U-Net.

![Autoencoder vs U-Net](https://raw.githubusercontent.com/RooTender/MineTexture/main/images/The-U-Net-vs-encoder-decoder-by-Jun-Yan-Zhu.png)



### Prerequisites
Before going to the details, let's install the necessary packages

In [ ]:
!pip install wandb

## Implementation
The code would basically rely on the typical GAN implementation that is constructed from:
- **Generator** - generates samples
- **Discriminator** - discriminates samples (*tells if it's ground truth or generated sample*)
- **"Random noise"** - it's the texture we want to transform into a certain style. The Generator using backward propagation learns the correlation between it and styled textures to create "fake image".
- **"Training data"** - ground truth based on which discriminator learns to find the patterns that differentiate the generated sample from the ground truth

![GAN Theory](https://raw.githubusercontent.com/RooTender/MineTexture/main/images/theailearner-generator-and-discriminator-by-kang-and-atul.png)

The implementation uses JAX. Compiled, modern and fast approach for scientific computations. The library (framework?) is currently maintained by Google and it's integration with Google Colab is just right.

### Generator
The generator code implements the possibility for tuning by using genetic algorithms.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class GeneratorConfig:
    def __init__(self,
                 num_layers, layer_multiplier, skip_connections,
                 dropout_rate, leaky_relu,
                 learning_rate, l1_lambda):
        self.num_layers = num_layers
        self.layer_multiplier = layer_multiplier
        self.skip_connections = skip_connections
        self.dropout_rate = dropout_rate
        self.relu_slope = leaky_relu
        self.learning_rate = learning_rate
        self.l1_lambda = l1_lambda

    def __str__(self):
        return (f"Number of Layers: {self.num_layers}\n"
                f"Layer Multiplier: {self.layer_multiplier}\n"
                f"Skip Connections: {self.skip_connections}\n"
                f"Dropout Rate: {self.dropout_rate}\n"
                f"Leaky ReLU: {self.relu_slope}\n"
                f"Learning Rate: {self.learning_rate}\n"
                f"L1 Lambda: {self.l1_lambda}")

In [ ]:
class Generator(nn.Module):
    def __init__(self, config):
        super(Generator, self).__init__()
        self.layer_multiplier = config.layer_multiplier
        self.relu_slope = config.relu_slope

        # Encoder
        self.encoder_layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(4, 64 * self.layer_multiplier, kernel_size=4, stride=2, padding=1),
                nn.LeakyReLU(self.relu_slope, inplace=True),
                nn.Conv2d(64 * self.layer_multiplier, 128 * self.layer_multiplier, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(128 * self.layer_multiplier)
            ),
            # Additional encoder layers can be defined here following the pattern.
        ])

        # Decoder
        self.decoder_layers = nn.ModuleList([
            nn.Sequential(
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(256 * self.layer_multiplier, 128 * self.layer_multiplier, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(128 * self.layer_multiplier),
                nn.Dropout(0.5, inplace=True)  # Adjust dropout rate as needed
            ),
            # Additional decoder layers can be defined here with consideration for skip connections and channel adjustments.
        ])

        # Final layer to adjust channels, assuming RGBA output.
        self.final_layer = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128 * self.layer_multiplier, 4, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        skip_connections = []

        # Encoder forward pass
        for layer in self.encoder_layers:
            x = layer(x)
            skip_connections.append(x)

        # Decoder forward pass with skip connections
        for i, layer in enumerate(self.decoder_layers):
            if i < len(skip_connections):
                x = torch.cat([x, skip_connections.pop()], dim=1)
            x = layer(x)

        out = self.final_layer(x)
        return out

### Discriminator
The discriminator code is simple, we **always** aim at the same approach. It performs well for smaller images and *in the future* it would also be tuned using genetic algorythms.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        self.structure = nn.Sequential(
            nn.Conv2d(4*2, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, kernel_size=4,
                      stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, original_images, target_images):
        combined_images = torch.cat((original_images, target_images), dim=1)
        return self.structure(combined_images)

# Data loading

## Loading image dataset
The necessary samples were extracted from the Minecraft textures and **paired** with the styled one's. Pix2Pix architecture relies on the concept of the comparison and focusing on certain domain knowledge. CycleGAN on the other hand, not.

### Data loader


Now we're implementing a way to load images for AI, where you can easily load input and output images from specified directories, making it useful for working with image datasets.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
class ImageLoaderDataset(Dataset):
    def __init__(self, input_root_dir, output_root_dir, transform=None):
        self.input_root_dir = input_root_dir
        self.output_root_dir = output_root_dir
        self.transform = transform
        self.image_paths = [(os.path.join(input_root_dir, file),
                             os.path.join(output_root_dir, file))
                            for file in os.listdir(input_root_dir)]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        input_image_path, output_image_path = self.image_paths[idx]
        input_image = Image.open(input_image_path)
        output_image = Image.open(output_image_path)

        input_image = input_image.resize(
            output_image.size, Image.Resampling.NEAREST)

        if self.transform:
            input_image = self.transform(input_image)
            output_image = self.transform(output_image)

        return input_image, output_image

In [ ]:
class DatasetLoader:
    def __init__(self, input_path: str, ground_truth_path: str):
        self.input_path = input_path
        self.output_path = ground_truth_path

    def load(self, batch_size: int):
        transformator = transforms.Compose([
            transforms.ToTensor()
        ])

        train_data = ImageLoaderDataset(
            os.path.join(self.input_path, 'train'),
            os.path.join(self.output_path, 'train'),
            transform=transformator)

        valid_data = ImageLoaderDataset(
            os.path.join(self.input_path, 'valid'),
            os.path.join(self.output_path, 'valid'),
            transform=transformator)

        train_loader = DataLoader(train_data, batch_size=batch_size,
                                  shuffle=False, pin_memory=True, num_workers=4)
        val_loader = DataLoader(valid_data, batch_size=batch_size,
                                shuffle=False, pin_memory=True, num_workers=4)

        return train_loader, val_loader

## Connecting to Google Drive storage

Attempts to connect to google drive in order to:

- download the necessary data
- save the best trained models

### Google Drive auth

In [ ]:
import os
from google.colab import drive
import shutil
import zipfile

In [ ]:
root = "/root"
os.makedirs(root, exist_ok=True)

drive.mount(f"{root}/gdrive", force_remount=True)

# Setup directory called "MineTexture" on GDrive account
!ln -sfn '/root/gdrive/My Drive/MineTexture' {root}/symdrive
os.makedirs(f"{root}/gdrive/My Drive/MineTexture", exist_ok=True)

Mounted at /root/gdrive


In [ ]:
drive = f'{root}/symdrive'
print(drive)

/root/symdrive


### Handling files


Create a storage for handling original and styled samples

In [ ]:
os.makedirs(f'{drive}/input', exist_ok=True)
os.makedirs(f'{drive}/target', exist_ok=True)

Create function that would decompress the packed zips

In [ ]:
def extract_zip_files(zip_directory, target_directory):
    for file in os.listdir(zip_directory):
        if file.endswith('.zip'):
            zip_file_path = os.path.join(zip_directory, file)
            target_zip_path = os.path.join(target_directory, file)

            # Copy the zip file to the target directory
            shutil.copy2(zip_file_path, target_zip_path)

            # Create a directory for the extracted files
            # The directory name is the zip file name without the '.zip' extension
            new_directory = os.path.splitext(file)[0]
            extract_path = os.path.join(target_directory, new_directory)

            if not os.path.exists(extract_path):
                os.makedirs(extract_path)

            print(f'Processing {new_directory}...')

            with zipfile.ZipFile(target_zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)

            os.remove(target_zip_path)

# Training
This is where the fun begins 🕺. Here we will train the model in order to generate our awesome Minecraft textures.

## Logic

### Loss functions

1. The **discriminator loss** function assesses how well the model distinguishes between real and fake images. It tries to make real images look distinct from fake ones.

2. The **generator loss** function evaluates how effectively the generator can trick the discriminator by generating images that are just as convincing as real ones. This improves the generator's skill at creating realistic images.

3. This **adversarial process** helps the generator refine its outputs to closely match the distribution of real images.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class GAN(nn.Module):
    def __init__(self, generator, discriminator, l1_lambda):
        super(GAN, self).__init__()
        self.generator = generator
        self.discriminator = discriminator
        self.l1_lambda = l1_lambda

    def discriminator_loss(self, original_images, target_images):
        fake_images = self.generator(original_images)

        pred_real = self.discriminator(original_images, target_images)
        pred_fake = self.discriminator(original_images, fake_images)

        real_labels = torch.ones_like(pred_real)  # Real images should be treated as 'real'
        fake_labels = torch.zeros_like(pred_fake) # Fake images should be treated as 'fake'

        loss_real = F.binary_cross_entropy_with_logits(pred_real, real_labels)
        loss_fake = F.binary_cross_entropy_with_logits(pred_fake, fake_labels)

        return (loss_real + loss_fake) / 2

    def generator_loss(self, original_images, target_images):
        fake_images = self.generator(original_images)

        pred_fake = self.discriminator(original_images, fake_images)
        real_labels = torch.ones_like(pred_fake)
        gen_loss = F.binary_cross_entropy_with_logits(pred_fake, real_labels)

        l1_loss = torch.mean(torch.abs(target_images - fake_images))

        return gen_loss + self.l1_lambda * l1_loss

### Train steps

In [ ]:
import wandb

In [ ]:
def train_step(generator, discriminator, gan, g_optimizer, d_optimizer, original_images, target_images):
    # Ensure generator and discriminator are in train mode
    generator.train()
    discriminator.train()

    # Zero the gradients on the optimizers
    g_optimizer.zero_grad()
    d_optimizer.zero_grad()

    # Compute generator loss
    g_loss = gan.generator_loss(original_images, target_images)
    g_loss.backward()
    g_optimizer.step()

    # Compute discriminator loss
    d_loss = gan.discriminator_loss(original_images, target_images)
    d_loss.backward()
    d_optimizer.step()

    return g_loss.item(), d_loss.item()

In [ ]:
def valid_step(generator, discriminator, gan, original_images, target_images):
    # Ensure generator and discriminator are in evaluation mode
    generator.eval()
    discriminator.eval()

    # Disable gradient calculations
    with torch.no_grad():
        g_loss = gan.generator_loss(original_images, target_images)
        d_loss = gan.discriminator_loss(original_images, target_images)

    return g_loss.item(), d_loss.item()

In [ ]:
def train(generator_hyperparameters, input_path, target_path, epochs, batch_size):
    wandb.init(project="MineTexture", entity="RooTender")
    wandb.config = {
        "learning_rate": generator_hyperparameters.learning_rate,
        "epochs": epochs,
        "batch_size": batch_size,
        "l1_lambda": generator_hyperparameters.l1_lambda,
    }

    # Initialize device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # Initialize models
    generator = Generator(generator_hyperparameters).to(device)
    discriminator = Discriminator().to(device)

    # GAN loss module
    gan = GAN(generator, discriminator, wandb.config['l1_lambda'])

    # Initialize optimizers
    g_optimizer = torch.optim.Adam(generator.parameters(), lr=wandb.config['learning_rate'])
    d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.002)

    # Prepare data loaders
    data_loader = DatasetLoader(input_path, target_path)
    train_loader, val_loader = data_loader.load(batch_size)

    valid_loss = 0

    print(generator_hyperparameters)

    # Training loop
    for epoch in range(epochs):
        total_g_train_loss = 0
        total_d_train_loss = 0
        train_batches = 0

        total_g_valid_loss = 0
        total_d_valid_loss = 0
        valid_batches = 0

        # Training
        for (original_images, target_images) in train_loader:
            original_images = original_images.to(device)
            target_images = target_images.to(device)

            # Train step
            g_loss, d_loss = train_step(generator, discriminator, gan, g_optimizer, d_optimizer, original_images, target_images)
            total_g_train_loss += g_loss
            total_d_train_loss += d_loss

            train_batches += 1

        g_train_loss = total_g_train_loss / train_batches
        d_train_loss = total_d_train_loss / train_batches

        # Validation
        for (original_images, target_images) in val_loader:
            original_images = original_images.to(device)
            target_images = target_images.to(device)

            # Validation step
            g_loss, d_loss = valid_step(generator, discriminator, gan, original_images, target_images)
            total_g_valid_loss += g_loss
            total_d_valid_loss += d_loss

            valid_batches += 1

        g_valid_loss = total_g_valid_loss / valid_batches
        d_valid_loss = total_d_valid_loss / valid_batches

        # Log metrics to wandb
        wandb.log({
            "Training Generator loss": g_train_loss,
            "Training Discriminator loss": d_train_loss,
            "Validation Generator loss": g_valid_loss,
            "Validation Discriminator loss": d_valid_loss
        })

        print(".", end="")
        valid_loss = g_valid_loss

    print('')
    return valid_loss

## Execution
First of all, let's load the datasets. I guess the names are speaking for themselves. The images must have:
- Equal size (therefore the `scale_factor` was introduced.)
- The same amount of data

### Prepare the data

Create directories for temporary data storage

In [ ]:
os.makedirs('data/input', exist_ok=True)
os.makedirs('data/target', exist_ok=True)

Extract the files stored on your drive to the colab environment

In [ ]:
extract_zip_files(f'{drive}/input', 'data/input')
extract_zip_files(f'{drive}/target', 'data/target')

Processing 8x8...
Processing 8x8_prepared...
Processing 16x16...
Processing 16x16_prepared...


### GA-optimized Neural Network 🧬

The upper example was a taste of how the GAN network that was build works. However, the performance of training highly depends on the choice of hyperparameters.

Here is introduced a genetic algorithm that crossovers the attributes that had the greatest impact on the network performance.

#### Genetic operations
The we have to define the operations that would allow us for proper genetic evolution.

In [ ]:
import random

In [ ]:
def create_individual(space):
    return {key: random.choice(values) for key, values in space.items()}

def initialize_population(size, space):
    return [create_individual(space) for _ in range(size)]

In [ ]:
def fitness(individual, input_path, target_path, epochs, batch_size):
    config = GeneratorConfig(**individual)
    return train(config, input_path, target_path, epochs, batch_size)

In [ ]:
def selection(population, scores, k=3):
    selection_ix = random.randint(0, len(population)-1)
    for ix in random.sample(range(len(population)), k-1):
        if scores[ix] < scores[selection_ix]:
            selection_ix = ix
    return population[selection_ix]

In [ ]:
def crossover(parent1, parent2, rate=0.5):
    if random.random() < rate:
        child1, child2 = {}, {}
        for key in parent1:
            if random.random() < 0.5:
                child1[key] = parent1[key]
                child2[key] = parent2[key]
            else:
                child1[key] = parent2[key]
                child2[key] = parent1[key]
        return [child1, child2]
    return [parent1, parent2]

In [ ]:
def mutate(individual, space, rate=0.1):
    for key, values in space.items():
        if random.random() < rate:
            individual[key] = random.choice(values)

#### Hyperparameters space
First we shall define the space that would be a boundary for matching our perfect combination

In [ ]:
hyperparameter_space = {
    'num_layers': [2, 3, 4, 5],
    'layer_multiplier': [1, 2],
    'skip_connections': [0, 1, 2],
    'dropout_rate': [0.3, 0.4, 0.5, 0.6, 0.7],
    'leaky_relu': [0.1, 0.2, 0.3],
    'learning_rate': [0.001, 0.002, 0.02],
    'l1_lambda': [10, 50, 100]
}

#### Genetic training
After defining everything it's time to put it all together. It's working, give it a try.

In [ ]:
population_size = 10
generations = 5
mutation_rate = 0.1
crossover_rate = 0.9

# Initialize population
population = initialize_population(population_size, hyperparameter_space)

for generation in range(generations):
    # Evaluate fitness
    scores = [fitness(individual, 'data/input/8x8_prepared', 'data/target/16x16_prepared', 50, 64) for individual in population]

    # Selection and creating the next generation
    selected = [selection(population, scores) for _ in range(population_size)]
    children = list()
    for i in range(0, population_size, 2):
        p1, p2 = selected[i], selected[i+1]
        for c in crossover(p1, p2, crossover_rate):
            mutate(c, hyperparameter_space, mutation_rate)
            children.append(c)
    population = children

    # Optionally print best score here
    best_score = min(scores)
    print(f'Generation {generation + 1}, Best Score: {best_score}')

# Find best solution at the end
best_index = scores.index(min(scores))
best_individual = population[best_index]
print('Best Hyperparameters:', best_individual)

Problem at: <ipython-input-57-4955539fd39c> 2 train


Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/wandb_init.py", line 1176, in init
    run = wi.init()
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/wandb_init.py", line 621, in init
    manager._inform_init(
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/wandb_manager.py", line 200, in _inform_init
    svc_iface._svc_inform_init(settings=settings, run_id=run_id)
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/service/service_sock.py", line 39, in _svc_inform_init
    self._sock_client.send(inform_init=inform_init)
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/lib/sock_client.py", line 211, in send
    self.send_server_request(server_req)
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/lib/sock_client.py", line 155, in send_server_request
    self._send_message(msg)
  File "/usr/local/lib/python3.10/dist-packages/wandb/sdk/lib/sock_client.py", line 152, in _send_message
    self._sendall

Error: An unexpected error occurred

### Manual training 🏋️‍♂️

After you found out the best hyperparameters, it's time to setup the network with them on your own.

The best model will be saved. Have fun 👋!

In [ ]:
# Setup
#generator_hyperparameters = GeneratorConfig(num_layers=3, layer_multiplier=1, skip_connections=1, dropout_rate=0.5, leaky_relu=0.2)

# Training
#train(generator_hyperparameters, 'data/input/8x8_prepared', 'data/target/16x16_prepared', 50, 16, 100)